####======================================================
### **Homework Assignment 10**
### Saurav Luthra, 55027009
### **ECE 648 - Machine Learning**

#### ====================================================
#### PREPROCESSING DATA
#### ======================================================

In [ ]:
# Code adapted from 'chp15rnn_LanguageModel.ipynb'

import numpy as np
import torch.nn as nn
import torch
from torch.utils.data import Dataset

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## Reading and processing text
with open('/content/drive/My Drive/ECE648/1268-0.txt', 'r', encoding="utf8") as fp:
    text=fp.read()

start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find('End of the Project Gutenberg')

text = text[start_indx:end_indx]
char_set = set(text)
print('Total Length:', len(text))
print('Unique Characters:', len(char_set))
chars_sorted = sorted(char_set)
char2int = {ch:i for i,ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)

text_encoded = np.array(
    [char2int[ch] for ch in text],
    dtype=np.int32)

print('Text encoded shape: ', text_encoded.shape)

print(text[:15], '     == Encoding ==> ', text_encoded[:15])
print(text_encoded[15:21], ' == Reverse  ==> ', ''.join(char_array[text_encoded[15:21]]))
for ex in text_encoded[:5]:
    print('{} -> {}'.format(ex, char_array[ex]))
seq_length = 40
chunk_size = seq_length + 1

text_chunks = [text_encoded[i:i+chunk_size]
               for i in range(len(text_encoded)-chunk_size+1)]

## inspection:
for seq in text_chunks[:1]:
    input_seq = seq[:seq_length]
    target = seq[seq_length]
    print(input_seq, ' -> ', target)
    print(repr(''.join(char_array[input_seq])),
          ' -> ', repr(''.join(char_array[target])))

class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long()

seq_dataset = TextDataset(torch.tensor(text_chunks))
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input (x):', repr(''.join(char_array[seq])))
    print('Target (y):', repr(''.join(char_array[target])))
    print()
    if i == 1:
        break
#device = torch.device("cuda:0")
device = 'cpu'
from torch.utils.data import DataLoader

batch_size = 64

torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)


Total Length: 1112350
Unique Characters: 80
Text encoded shape:  (1112350,)
THE MYSTERIOUS       == Encoding ==>  [44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]
[33 43 36 25 38 28]  == Reverse  ==>  ISLAND
44 -> T
32 -> H
29 -> E
1 ->  
37 -> M
[44 32 29  1 37 48 43 44 29 42 33 39 45 43  1 33 43 36 25 38 28  1  6  6
  6  0  0  0  0  0 40 67 64 53 70 52 54 53  1 51]  ->  74
'THE MYSTERIOUS ISLAND ***\n\n\n\n\nProduced b'  ->  'y'
 Input (x): 'THE MYSTERIOUS ISLAND ***\n\n\n\n\nProduced b'
Target (y): 'HE MYSTERIOUS ISLAND ***\n\n\n\n\nProduced by'

 Input (x): 'HE MYSTERIOUS ISLAND ***\n\n\n\n\nProduced by'
Target (y): 'E MYSTERIOUS ISLAND ***\n\n\n\n\nProduced by '



<ipython-input-2-dd69b0ad1480>:51: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  seq_dataset = TextDataset(torch.tensor(text_chunks))


#### ====================================================
#### 1 - LSTM LAYER
#### ======================================================

In [28]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size,
                           batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden, cell

    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size)
        return hidden.to(device), cell.to(device)


In [29]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512

torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size)
model = model.to(device)
model

RNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [30]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

num_epochs = 10000

torch.manual_seed(1)

In [ ]:
for epoch in range(num_epochs):
    hidden, cell = model.init_hidden(batch_size)
    seq_batch, target_batch = next(iter(seq_dl))
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)
    optimizer.zero_grad()
    loss = 0
    for c in range(seq_length):
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])
    loss.backward()
    optimizer.step()
    loss = loss.item()/seq_length
    if epoch % 500 == 0:
        print(f'Epoch {epoch} loss: {loss:.4f}')


Epoch 0 loss: 4.3722
Epoch 500 loss: 1.3878
Epoch 1000 loss: 1.3424
Epoch 1500 loss: 1.2189
Epoch 2000 loss: 1.2284
Epoch 2500 loss: 1.2042
Epoch 3000 loss: 1.1477
Epoch 3500 loss: 1.1280
Epoch 4000 loss: 1.1706
Epoch 4500 loss: 1.1514
Epoch 5000 loss: 1.0885
Epoch 5500 loss: 1.1272
Epoch 6000 loss: 1.1526
Epoch 6500 loss: 1.1158
Epoch 7000 loss: 1.1071
Epoch 7500 loss: 1.1711
Epoch 8000 loss: 1.1402
Epoch 8500 loss: 1.1454
Epoch 9000 loss: 1.1042
Epoch 9500 loss: 1.0958


In [1]:
def sample(model, starting_str,
           len_generated_text=500,
           scale_factor=1.0):

    encoded_input = torch.tensor([char2int[s] for s in starting_str])
    encoded_input = torch.reshape(encoded_input, (1, -1))

    generated_str = starting_str

    model.eval()
    hidden, cell = model.init_hidden(1)
    hidden = hidden.to('cpu')
    cell = cell.to('cpu')
    for c in range(len(starting_str)-1):
        _, hidden, cell = model(encoded_input[:, c].view(1), hidden, cell)

    last_char = encoded_input[:, -1]
    for i in range(len_generated_text):
        logits, hidden, cell = model(last_char.view(1), hidden, cell)
        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor

        # changed: This makes the model always pick the most likely next character
        #m = Categorical(logits=scaled_logits)
        #last_char = m.sample()
        last_char = torch.argmax(scaled_logits, dim=-1)

        generated_str += str(char_array[last_char])

    return generated_str


In [ ]:
torch.manual_seed(1)
model.to('cpu')
print(sample(model, starting_str='The island'))

In [31]:
def evaluate_accuracy(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for seq_batch, target_batch in data_loader:
            hidden, cell = model.init_hidden(seq_batch.size(0))
            seq_batch = seq_batch.to(device)
            target_batch = target_batch.to(device)
            for c in range(seq_length):
                pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
                pred_char = torch.argmax(pred, dim=1)
                correct += (pred_char == target_batch[:, c]).sum().item()
                total += pred_char.size(0)
    return correct / total


In [35]:
print('Accuracy:', evaluate_accuracy(model, seq_dl))

Accuracy: 0.6465613132228551


#### ====================================================
#### 2 - RNN LAYER
#### ======================================================

In [ ]:
class RNN1(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.RNN(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden):
        out = self.embedding(x).unsqueeze(1)
        out, hidden = self.rnn(out, hidden)
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden

    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
        return hidden.to(device)


In [ ]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512

torch.manual_seed(1)
model1 = RNN1(vocab_size, embed_dim, rnn_hidden_size)
model1 = model1.to(device)
model1

RNN1(
  (embedding): Embedding(80, 256)
  (rnn): RNN(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [ ]:
loss_fn1 = nn.CrossEntropyLoss()
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.005)

num_epochs = 5000

torch.manual_seed(1)

In [ ]:
for epoch in range(num_epochs):
    hidden = model1.init_hidden(batch_size)
    seq_batch, target_batch = next(iter(seq_dl))
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)
    optimizer1.zero_grad()
    loss = 0
    for c in range(seq_length):
        pred, hidden = model1(seq_batch[:, c], hidden)
        loss += loss_fn1(pred, target_batch[:, c])
    loss.backward()
    optimizer1.step()
    loss = loss.item() / seq_length
    if epoch % 500 == 0:
        print(f'Epoch {epoch} loss: {loss:.4f}')


Epoch 0 loss: 4.3848
Epoch 500 loss: 1.5231
Epoch 1000 loss: 1.5408
Epoch 1500 loss: 1.4378
Epoch 2000 loss: 1.4354
Epoch 2500 loss: 1.4420
Epoch 3000 loss: 1.4080
Epoch 3500 loss: 1.3856
Epoch 4000 loss: 1.4950
Epoch 4500 loss: 1.4846


In [ ]:
torch.save(model1.state_dict(), '/content/drive/My Drive/ECE648/model1.pth')

In [ ]:
def sample1(model, starting_str, len_generated_text=500, scale_factor=1.0):
    encoded_input = torch.tensor([char2int[s] for s in starting_str])
    encoded_input = torch.reshape(encoded_input, (1, -1))

    generated_str = starting_str

    model.eval()

    # Check if model.init_hidden returns 1 or 2 values
    init_hidden_result = model.init_hidden(1)
    if isinstance(init_hidden_result, tuple):
        hidden, cell = init_hidden_result
        use_cell = True
    else:
        hidden = init_hidden_result
        use_cell = False

    hidden = hidden.to('cpu')
    if use_cell:
        cell = cell.to('cpu')

    for c in range(len(starting_str) - 1):
        if use_cell:
            _, hidden, cell = model(encoded_input[:, c].view(1), hidden, cell)
        else:
            _, hidden = model(encoded_input[:, c].view(1), hidden)

    last_char = encoded_input[:, -1]

    for i in range(len_generated_text):
        if use_cell:
            logits, hidden, cell = model(last_char.view(1), hidden, cell)
        else:
            logits, hidden = model(last_char.view(1), hidden)

        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor

        # Use most likely character
        last_char = torch.argmax(scaled_logits, dim=-1)

        generated_str += str(char_array[last_char])

    return generated_str


In [ ]:
torch.manual_seed(1)
model1.to('cpu')
print(sample1(model1, starting_str='The island'))

The island, and the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists was a supply of the colonists 


In [ ]:
def evaluate_accuracy1(model, data_loader, seq_length):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for seq_batch, target_batch in data_loader:
            if isinstance(model.rnn, nn.LSTM):
                hidden, cell = model.init_hidden(seq_batch.size(0))
                hidden = hidden.to(device)
                cell = cell.to(device)
                for c in range(seq_length):
                    pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
            else:
                hidden = model.init_hidden(seq_batch.size(0))
                hidden = hidden.to(device)
                for c in range(seq_length):
                    pred, hidden = model(seq_batch[:, c], hidden)

            pred_char = torch.argmax(pred, dim=1)
            correct += (pred_char == target_batch[:, c]).sum().item()
            total += pred_char.size(0)

    return correct / total


In [ ]:
print('Accuracy:', evaluate_accuracy1(model1, seq_dl, seq_length))


Accuracy: 0.567960973013407


#### ====================================================
#### 3 - RNN LAYER (x2)
#### ======================================================

In [16]:
class RNN2(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.num_layers = 2  # NEW: number of stacked RNN layers
        self.rnn = nn.RNN(embed_dim, rnn_hidden_size, num_layers=self.num_layers, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden):
        out = self.embedding(x).unsqueeze(1)
        out, hidden = self.rnn(out, hidden)
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden

    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers, batch_size, self.rnn_hidden_size)
        return hidden.to(device)


In [17]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512

torch.manual_seed(1)
model2 = RNN2(vocab_size, embed_dim, rnn_hidden_size)
model2 = model2.to(device)
model2

RNN2(
  (embedding): Embedding(80, 256)
  (rnn): RNN(256, 512, num_layers=2, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [18]:
loss_fn2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.005)

num_epochs = 5000

torch.manual_seed(1)

In [22]:
for epoch in range(num_epochs):
    hidden = model2.init_hidden(batch_size)
    seq_batch, target_batch = next(iter(seq_dl))
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)
    optimizer2.zero_grad()
    loss = 0
    for c in range(seq_length):
        pred, hidden = model2(seq_batch[:, c], hidden)
        loss += loss_fn2(pred, target_batch[:, c])
    loss.backward()
    optimizer2.step()
    loss = loss.item() / seq_length
    if epoch % 500 == 0:
        print(f'Epoch {epoch} loss: {loss:.4f}')


Epoch 0 loss: 4.3711
Epoch 500 loss: 1.5483
Epoch 1000 loss: 1.5567
Epoch 1500 loss: 1.4901
Epoch 2000 loss: 1.4921
Epoch 2500 loss: 1.4964
Epoch 3000 loss: 1.4693
Epoch 3500 loss: 1.4455
Epoch 4000 loss: 1.5164
Epoch 4500 loss: 1.5508


In [26]:
torch.save(model2.state_dict(), '/content/drive/My Drive/ECE648/model2.pth')

In [21]:
def sample2(model, starting_str, len_generated_text=500, scale_factor=1.0):
    encoded_input = torch.tensor([char2int[s] for s in starting_str])
    encoded_input = torch.reshape(encoded_input, (1, -1))

    generated_str = starting_str

    model.eval()
    hidden = model.init_hidden(1)
    hidden = hidden.to('cpu')

    for c in range(len(starting_str) - 1):
        _, hidden = model(encoded_input[:, c].view(1), hidden)

    last_char = encoded_input[:, -1]
    for i in range(len_generated_text):
        logits, hidden = model(last_char.view(1), hidden)
        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor

        # Pick most likely next character
        last_char = torch.argmax(scaled_logits, dim=-1)

        generated_str += str(char_array[last_char])

    return generated_str


In [27]:

torch.manual_seed(1)
model2.to('cpu')
print(sample2(model2, starting_str='The island'))


The island, and the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway the castaway 


In [24]:
def evaluate_accuracy2(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for seq_batch, target_batch in data_loader:
            hidden = model.init_hidden(seq_batch.size(0))
            seq_batch = seq_batch.to(device)
            target_batch = target_batch.to(device)
            for c in range(seq_length):
                pred, hidden = model(seq_batch[:, c], hidden)
                pred_char = torch.argmax(pred, dim=1)
                correct += (pred_char == target_batch[:, c]).sum().item()
                total += pred_char.size(0)
    return correct / total


In [25]:
print('Accuracy:', evaluate_accuracy2(model2, seq_dl))

Accuracy: 0.5410555663444387
